# APEX — Stage 1: Ingest

**Goal of this stage:** download every 2023 *qualifying* session from FastF1, stack all the laps into one table, save it, and eyeball what we got.

No cleaning or modelling yet — we just want the raw data in front of us.

### 1. Imports & cache
FastF1 talks to the official F1 timing servers. The **cache** means each session is downloaded once and read from disk forever after.

In [ ]:
import fastf1                       # the data source (official F1 timing)
import pandas as pd                 # tables (DataFrames)
from pathlib import Path            # OS-safe file paths


# Walk up the folder tree until we find WHITEPAPER.md, so this notebook works
# no matter which directory it's run from.
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while p != p.parent:                     # stop at the filesystem root
        if (p / "WHITEPAPER.md").exists():
            return p
        p = p.parent
    raise RuntimeError("repo root not found")


REPO = find_repo_root(Path.cwd())
CACHE = REPO / "data" / "cache"         # FastF1's download cache
PARQUET = REPO / "data" / "parquet"     # where we save our own tables
CACHE.mkdir(parents=True, exist_ok=True)
PARQUET.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE))   # first run downloads; later runs read locally
print("Repo root:", REPO)

### 2. The season schedule
Every race weekend in 2023. `include_testing=False` drops pre-season testing (not competitive sessions).

In [ ]:
SEASON = 2023

schedule = fastf1.get_event_schedule(SEASON, include_testing=False)
schedule[["RoundNumber", "EventName", "Country", "EventDate"]]

### 3. Pull every qualifying session
For each round we load the `'Q'` session and copy out its lap table. We load laps + weather + race-control messages but **not telemetry** (the per-metre car-sensor data) — we don't need it and it's ~100× larger. Each lap is tagged with its season/round/event so we never lose track of which session it came from.

In [ ]:
frames = []
for rnd, name in zip(schedule["RoundNumber"], schedule["EventName"]):
    session = fastf1.get_session(SEASON, rnd, "Q")   # the qualifying session object
    session.load(telemetry=False)                    # downloads on first run, cached after
    laps = session.laps.copy()                       # this session's lap table
    laps["Season"] = SEASON                           # tag so we can tell sessions apart
    laps["Round"] = rnd
    laps["EventName"] = name
    frames.append(laps)
    print(f"R{rnd:>2}  {name:<30} {len(laps):>4} laps")

# Stack all sessions into one plain DataFrame (one row per lap).
laps_raw = pd.DataFrame(pd.concat(frames, ignore_index=True))
print("\nTOTAL:", laps_raw.shape[0], "laps x", laps_raw.shape[1], "columns")

### 4. Save the raw laps
Write to Parquet (a fast, compact table format) so we can experiment later without re-downloading.

In [ ]:
out_path = PARQUET / f"laps_q_{SEASON}_raw.parquet"
laps_raw.to_parquet(out_path)
print("Saved:", out_path.relative_to(REPO))

### 5. Eyeball what we got
A few key columns for the first handful of laps. `LapTime` is a duration; `TrackStatus`, `Deleted` and `IsPersonalBest` are the flags we'll use to clean the data in the next stage.

In [ ]:
cols = ["EventName", "Driver", "Team", "LapNumber", "LapTime",
        "Compound", "TrackStatus", "Deleted", "IsPersonalBest"]
laps_raw[cols].head(12)